# Chemical simulation interface
This notebook provides an interface for a toy model of a chemical reaction in the gas phase.

## Import libraries

In [1]:
# %pip install -r requirements.txt

In [1]:
%load_ext autoreload
%autoreload 2

In [27]:
%pip install -r requirements.txt

In [4]:
from collider import Simulation
from ipycanvas import Canvas

In [29]:
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from ipycanvas import Canvas, hold_canvas
import ipywidgets as widgets
import time

## Initialize and run the simulation
Initialize the Simulation class with the desired parameters and run the simulation loop.

In [30]:
simulation = Simulation(Canvas, 800, 600, 100, temperature=100)
simulation.canvas



Canvas(height=600, width=800)

In [31]:
simulation.run()

KeyboardInterrupt: 

In [32]:
print(simulation.get_current_metrics())

{'collisions': 10522760}


In [33]:
# Final version with live-updating chart with collision count

from collider import Simulation
from ipycanvas import Canvas, hold_canvas
from ipywidgets import VBox
import time

# 1. Initialize simulation and separate chart canvas
simulation = Simulation(Canvas, 800, 600, 100, temperature=100)
chart_canvas = Canvas(width=800, height=250)

# Display both canvases together
display(VBox([simulation.canvas, chart_canvas]))

max_points = 300
step = 0

while True:
    # 2. Step simulation and draw particles
    simulation.update()
    simulation.draw()
    
    # 3. Update chart every 5 steps to keep animation completely lag-free
    if step % 5 == 0:
        data = simulation.collision_history[-max_points:]
        
        with hold_canvas(chart_canvas):
            chart_canvas.clear()
            
            # Axes
            chart_canvas.stroke_style = "black"
            chart_canvas.line_width = 1
            chart_canvas.stroke_line(60, 20, 60, 210)   # Y-axis
            chart_canvas.stroke_line(60, 210, 770, 210) # X-axis
            
            # Title & Axis Labels
            chart_canvas.fill_style = "black"
            chart_canvas.font = "14px sans-serif"
            chart_canvas.fill_text(f"Collision Count: {simulation.collision_count}", 70, 25)
            chart_canvas.fill_text("Simulation Steps", 370, 240)
            
            chart_canvas.save()
            chart_canvas.translate(20, 150)
            chart_canvas.rotate(-1.5708)
            chart_canvas.fill_text("Collisions", 0, 0)
            chart_canvas.restore()
            
            # Line plot
            if len(data) > 1:
                min_y = min(data)
                max_y = max(data)
                y_range = (max_y - min_y) if max_y > min_y else 1
                
                chart_canvas.stroke_style = "blue"
                chart_canvas.line_width = 2
                chart_canvas.begin_path()
                
                for i, val in enumerate(data):
                    px = 60 + (i / max_points) * 710
                    py = 210 - ((val - min_y) / y_range) * 180
                    if i == 0:
                        chart_canvas.move_to(px, py)
                    else:
                        chart_canvas.line_to(px, py)
                chart_canvas.stroke()
                
    step += 1
    time.sleep(0.002)

KeyboardInterrupt: 